# BrandIQ — Customer Data Preprocessing & Feature Engineering

This Jupyter Notebook documents the preprocessing and feature engineering pipeline for the D2C fashion brand customer dataset (`Dataset.csv`).
The formulas and normalization steps engineered here are ported directly to JavaScript for real-time customer data uploads and manual entry modifications.

In [ ]:
import pandas as pd
import numpy as np
import json

# Load dataset
df = pd.read_csv('Dataset.csv')
print(f"Loaded {len(df)} customer records.")
df.head()

## 1. Clean Data
Standardize string columns, clean numerical columns, and fill missing items.

In [ ]:
# Strip whitespace
string_cols = df.select_dtypes(include=['object']).columns
for col in string_cols:
    df[col] = df[col].str.strip()

# Set types
df['Age'] = df['Age'].astype(int)
df['Purchase Amount (USD)'] = df['Purchase Amount (USD)'].astype(float)
df['Review Rating'] = df['Review Rating'].astype(float)
df['Previous Purchases'] = df['Previous Purchases'].astype(int)

print("Data Types:")
print(df.dtypes)

## 2. Feature Engineering Rules
We compute the following properties:
1. **Frequency Score**: Maps purchase frequency (e.g. Weekly = 7, Annually = 1)
2. **Promo Dependency Score**: Average of Discount Applied (0/1) and Promo Code Used (0/1)
3. **Subscriber**: 1 if Subscription Status is Yes else 0
4. **Loyalty V1 Score**: `frequency_score_norm * 0.4 + previous_purchases_norm * 0.4 + subscriber * 0.2`
5. **Loyalty V2 Score**: `spend_norm * 0.35 + prev_norm * 0.35 + rating_norm * 0.2 + (1 - promo_dependency) * 0.1`
6. **Composite Value**: `loyalty_score + spend_norm`
7. **Value Tier**: Quartile of Composite Value (Bronze, Silver, Gold, Platinum)

In [ ]:
# 1. Frequency Score Mapping
freq_map = {
    'Weekly': 7,
    'Fortnightly': 6,
    'Bi-Weekly': 5,
    'Monthly': 4,
    'Quarterly': 3,
    'Every 3 Months': 2,
    'Annually': 1
}
df['frequency_score'] = df['Frequency of Purchases'].map(freq_map).fillna(4)

# 2. Promo Dependency
df['discount_used'] = df['Discount Applied'].apply(lambda x: 1 if str(x).lower() == 'yes' else 0)
df['promo_used'] = df['Promo Code Used'].apply(lambda x: 1 if str(x).lower() == 'yes' else 0)
df['promo_dependency_score'] = (df['discount_used'] + df['promo_used']) / 2

# 3. Subscriber Status
df['subscriber'] = df['Subscription Status'].apply(lambda x: 1 if str(x).lower() == 'yes' else 0)

# Normalizations
df['previous_purchases_norm'] = (df['Previous Purchases'] - df['Previous Purchases'].min()) / (df['Previous Purchases'].max() - df['Previous Purchases'].min())
df['spend_norm'] = (df['Purchase Amount (USD)'] - df['Purchase Amount (USD)'].min()) / (df['Purchase Amount (USD)'].max() - df['Purchase Amount (USD)'].min())
df['rating_norm'] = (df['Review Rating'] - df['Review Rating'].min()) / (df['Review Rating'].max() - df['Review Rating'].min())
df['frequency_norm'] = df['frequency_score'] / 7

# 4. Loyalty V1 and V2
df['loyalty_v1'] = df['frequency_norm'] * 0.4 + df['previous_purchases_norm'] * 0.4 + df['subscriber'] * 0.2
df['loyalty_v2'] = df['spend_norm'] * 0.35 + df['previous_purchases_norm'] * 0.35 + df['rating_norm'] * 0.2 + (1 - df['promo_dependency_score']) * 0.1

# Loyalty score uses v1
df['loyalty_score'] = df['loyalty_v1']

# 5. Composite Value
df['composite_value'] = df['loyalty_score'] + df['spend_norm']

# 6. Value Tier (Quartiles)
df['value_tier'] = pd.qcut(df['composite_value'], q=4, labels=['Bronze', 'Silver', 'Gold', 'Platinum'])

# Secondary Flags
df['satisfaction_flag'] = df['Review Rating'] >= 4.0
df['high_value_no_promo'] = ((df['value_tier'] == 'Gold') | (df['value_tier'] == 'Platinum')) & (df['promo_dependency_score'] == 0)

median_prev = df['Previous Purchases'].median()
df['promo_trap'] = (df['promo_dependency_score'] == 1) & (df['Previous Purchases'] < median_prev)
df['spend_efficiency'] = df['Purchase Amount (USD)'] / (df['Previous Purchases'] + 1)
df['churn_risk'] = (df['frequency_score'] <= 2) & (df['Review Rating'] < 3.5) & (df['promo_dependency_score'] >= 0.5)

# Age Groups
bins = [0, 25, 35, 45, 55, 120]
labels = ['18-25', '26-35', '36-45', '46-55', '56+']
df['age_group'] = pd.cut(df['Age'], bins=bins, labels=labels)

print("Feature Engineering Complete.")
df[['Customer ID', 'value_tier', 'loyalty_score', 'promo_dependency_score', 'churn_risk']].head()

## 3. Export to JSON
Format the engineered dataset matching the exact shape expected by the React context.

In [ ]:
# Mapping keys to matching Javascript naming
json_records = []
for _, row in df.iterrows():
    record = {
        'customer_id': int(row['Customer ID']),
        'age': int(row['Age']),
        'gender': str(row['Gender']),
        'item_purchased': str(row['Item Purchased']),
        'category': str(row['Category']),
        'purchase_amount': float(row['Purchase Amount (USD)']),
        'location': str(row['Location']),
        'size': str(row['Size']),
        'color': str(row['Color']),
        'season': str(row['Season']),
        'review_rating': float(row['Review Rating']),
        'subscription_status': str(row['Subscription Status']),
        'shipping_type': str(row['Shipping Type']),
        'discount_applied': str(row['Discount Applied']),
        'promo_code_used': str(row['Promo Code Used']),
        'previous_purchases': int(row['Previous Purchases']),
        'payment_method': str(row['Payment Method']),
        'frequency_of_purchases': str(row['Frequency of Purchases']),
        'frequency_score': int(row['frequency_score']),
        'promo_dependency_score': float(row['promo_dependency_score']),
        'loyalty_score': float(row['loyalty_score']),
        'composite_value': float(row['composite_value']),
        'value_tier': str(row['value_tier']),
        'satisfaction_flag': bool(row['satisfaction_flag']),
        'high_value_no_promo': bool(row['high_value_no_promo']),
        'promo_trap': bool(row['promo_trap']),
        'spend_efficiency': float(row['spend_efficiency']),
        'churn_risk': bool(row['churn_risk']),
        'age_group': str(row['age_group'])
    }
    json_records.append(record)

with open('public/data/customer_features.json', 'w') as f:
    json.dump(json_records, f, indent=2)

print("Exported customer features successfully to public/data/customer_features.json")